# Transaction Validation Notebook

Use this notebook to validate and inspect data at each pipeline stage.

**Purpose:**
- Query raw_transactions, normalized_transactions, and categorized_transactions
- Check for parsing errors, missing dates, zero amounts
- Validate normalization quality (merchant names, amounts, dates)
- Spot-check categorization results

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from staging.database import get_connection
con = get_connection()
print('Connected to database.')

In [ ]:
# --- Pipeline counts ---
for table in ['raw_transactions', 'normalized_transactions', 'categorized_transactions']:
    count = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table}: {count} rows')

In [ ]:
# --- Sample raw transactions ---
import pandas as pd

df_raw = con.execute("""
    SELECT institution, source_file, page_number,
           transaction_date_raw, merchant_raw, amount_raw
    FROM raw_transactions
    LIMIT 20
""").df()
df_raw

In [ ]:
# --- Sample normalized transactions ---
df_norm = con.execute("""
    SELECT institution, transaction_date, merchant_normalized,
           original_amount, currency, is_credit
    FROM normalized_transactions
    ORDER BY transaction_date
    LIMIT 20
""").df()
df_norm

In [ ]:
# --- Check for NULL dates (parsing failures) ---
null_dates = con.execute("""
    SELECT COUNT(*) FROM normalized_transactions WHERE transaction_date IS NULL
""").fetchone()[0]
print(f'Records with NULL transaction_date: {null_dates}')

# --- Check for zero amounts ---
zero_amounts = con.execute("""
    SELECT COUNT(*) FROM normalized_transactions WHERE original_amount = 0
""").fetchone()[0]
print(f'Records with zero amount: {zero_amounts}')

In [ ]:
# --- Duplicate analysis ---
dupes = con.execute("""
    SELECT dedupe_hash, COUNT(*) as cnt
    FROM normalized_transactions
    WHERE dedupe_hash IS NOT NULL
    GROUP BY dedupe_hash
    HAVING COUNT(*) > 1
    ORDER BY cnt DESC
    LIMIT 10
""").df()
print(f'Duplicate groups: {len(dupes)}')
dupes

In [ ]:
# --- Distribution by institution ---
con.execute("""
    SELECT institution, COUNT(*) as count,
           ROUND(SUM(original_amount), 2) as total
    FROM normalized_transactions
    GROUP BY institution
""").df()

In [ ]:
# --- Monthly spend breakdown ---
con.execute("""
    SELECT
        strftime(transaction_date, '%Y-%m') as month,
        COUNT(*) as transactions,
        ROUND(SUM(CASE WHEN is_credit = FALSE THEN original_amount ELSE 0 END), 2) as charges,
        ROUND(SUM(CASE WHEN is_credit = TRUE THEN ABS(original_amount) ELSE 0 END), 2) as credits
    FROM normalized_transactions
    WHERE transaction_date IS NOT NULL
    GROUP BY month
    ORDER BY month
""").df()